# 01 - Inventory and profiling

Inventario delle fonti, verifica dei manifest e controllo delle dimensioni principali del master ICCU.

### Riproducibilità

Questo notebook documenta e verifica la fase di inventario e profiling della pipeline.

I dati sorgente originali sono conservati nella directory `data/raw/`, mentre le trasformazioni riproducibili sono implementate negli script della directory `scripts/`.

Il notebook permette di ispezionare le fonti, i metadati, il profiling e gli output prodotti dalla pipeline.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for c in candidates:
        if (c / "data" / "processed").exists() and (c / "metadata").exists():
            return c
    raise FileNotFoundError("Eseguire il notebook dalla root del repository o da notebooks/.")

ROOT = project_root()
ROOT

## Inventory delle fonti e degli artefatti di profiling

Il notebook utilizza i manifest e le tabelle di profiling prodotti dalla pipeline e permette di verificarne contenuto, struttura e coerenza con i file RAW conservati nel repository.

In [2]:
inventory = pd.read_csv(ROOT / "metadata/file_inventory.csv", dtype=str, keep_default_na=False)
manifest = pd.read_csv(ROOT / "metadata/source_manifest.csv", dtype=str, keep_default_na=False)
print(f"File inventariati: {len(inventory):,}")
print(f"Fonti nel manifest: {len(manifest):,}")
display(manifest)

File inventariati: 21
Fonti nel manifest: 8


,source_id,publisher,dataset_name,source_url,local_file,format,encoding,license,download_date,temporal_coverage,spatial_coverage,update_frequency,role_in_project,sha256,notes
0,ICCU_ANAGRAFE_20260908,ICCU – Istituto Centrale per il Catalogo Unico...,Anagrafe delle Biblioteche Italiane – Open Data,https://anagrafe.iccu.sbn.it/it/open-data/,data/raw/iccu/opendata.zip,"ZIP (JSON, CSV, XML)",UTF-8 / UTF-8-SIG,CC0 1.0,2026-09-08,snapshot 2026-09-08T14:11:15,Italia; include alcune sedi italiane all’estero,quotidiana,master biblioteche e dataset ICCU secondari,536aab4de1ea4dcdb7e078ca2694f107dd3bdfd68d17a5...,Archivio RAW originale incluso nel repository;...
1,ISTAT_POSAS_2019,Istat – Istituto nazionale di statistica,"Popolazione residente per età, sesso e stato c...",https://demo.istat.it/app/?i=POS,data/raw/istat/POSAS_2019_it_Tutti_i_file.zip,ZIP/CSV,UTF-8-SIG,CC BY 4.0,2026-09-08,2019-01-01,"Italia, comuni/province/regioni/ripartizioni",annuale,baseline demografica 2019,b5cf8df245fbf82fefbf2ac3b74f43cfe5c89e45df19ce...,Archivio RAW originale incluso nel repository;...
2,ISTAT_POSAS_2025,Istat – Istituto nazionale di statistica,"Popolazione residente per età, sesso e stato c...",https://demo.istat.it/app/?i=POS,data/raw/istat/POSAS_2025_it_Tutti_i_file.zip,ZIP/CSV,UTF-8-SIG,CC BY 4.0,2026-09-08,2025-01-01,"Italia, comuni/province/regioni/ripartizioni",annuale,popolazione corrente e geografia canonica 2025,4008c4ead71b00af99ea2fa39fde79bf49e2be0a8cba20...,Archivio RAW originale incluso nel repository;...
3,CULTURAL_ON_V2,MiBACT / CNR-ISTC STLab,Cultural-ON (Cultural ONtology),https://dati.beniculturali.it/cultural-ON/ITA....,data/external/cultural-ON.owl,OWL/RDFXML,UTF-8,CC BY 3.0 IT,2026-09-08,versione 2.0 – 2016-03-30,vocabolario/ontologia di dominio,,ontologia esterna riutilizzata nella modellazi...,b81643ef5438e69fda1ce1c7bc5927d7a2c01491c7c0dd...,Utilizzata per il riuso e l’allineamento dei c...
4,ICCU_RELEASE_NOTES_1_6_P1,ICCU,Anagrafe delle biblioteche italiane – Note di ...,https://anagrafe.iccu.sbn.it/it/informazioni/f...,data/external/note-di-rilascio-1.6 (pagina 1).png,PNG,binary,CC BY-NC-SA 3.0 IT (licenza predefinita dei co...,2026-09-08,versione formato 1.6,,,documentazione struttura/formato e contesto ev...,c3324ecf7ce4173c7665c044b329f7fec844be593703e7...,Riproduzione immagine della documentazione uff...
5,ICCU_RELEASE_NOTES_1_6_P2,ICCU,Anagrafe delle biblioteche italiane – Note di ...,https://anagrafe.iccu.sbn.it/it/informazioni/f...,data/external/note-di-rilascio-1.6 (pagina 2).png,PNG,binary,CC BY-NC-SA 3.0 IT (licenza predefinita dei co...,2026-09-08,versione formato 1.6,,,documentazione struttura/formato e contesto ev...,f6d0d2089134049cf904447fd7d0547f52f18fcba1b284...,Riproduzione immagine della documentazione uff...
6,ICCU_RELEASE_NOTES_1_6_P3,ICCU,Anagrafe delle biblioteche italiane – Note di ...,https://anagrafe.iccu.sbn.it/it/informazioni/f...,data/external/note-di-rilascio-1.6 (pagina 3).png,PNG,binary,CC BY-NC-SA 3.0 IT (licenza predefinita dei co...,2026-09-08,versione formato 1.6,,,documentazione struttura/formato e contesto ev...,a92c8ff78b6ae12db0938f4e16bc32819bedfabf7651c6...,Riproduzione immagine della documentazione uff...
7,ICCU_RELEASE_NOTES_1_6_P4,ICCU,Anagrafe delle biblioteche italiane – Note di ...,https://anagrafe.iccu.sbn.it/it/informazioni/f...,data/external/note-di-rilascio-1.6 (pagina 4).png,PNG,binary,CC BY-NC-SA 3.0 IT (licenza predefinita dei co...,2026-09-08,versione formato 1.6,,,documentazione struttura/formato e contesto ev...,5168bf2a3e84e5300a1dc6e8321c14fc5420518ceacfe9...,Riproduzione immagine della documentazione uff...


In [3]:
profile = pd.read_csv(ROOT / "reports/data_profile_tables/processed_resource_profile.csv")
profile

,file,records,columns,sha256
0,analysis_municipality.csv,7896,33,264bb332c87ffae5288aec756eed69472f19d4f81b97ee...
1,library.csv,19611,31,201472af8b1cd0f74e465177106d5d744662af9e2a0af6...
2,library_contact.csv,62804,5,dbebd0779d4b77926be8adf384356b763a76829ccd27d9...
3,library_holdings.csv,93512,7,9fb4cb508c628e1488547e31307c98e8f60f05d3efa729...
4,library_mergers.csv,1502,7,1a34e43067851af150be203a2675fe0a2504b0eb596326...
5,library_previous_name.csv,9507,3,726fcf4cf59248fc70cd3e5fac5af17f5131848a432a32...
6,library_status.csv,19611,7,f5c61f143f7e6a938129bbb2eaab5e1ee54ab27f6b6ac4...
7,library_type.csv,13715,5,91e908d6189d8646ed2ba040725a8e420034e1aa8f88ea...
8,municipality_population.csv,7896,15,eb37a0011f6343756ef81d145d9418d25460f068f24d8e...
9,special_collection.csv,9737,19,d84c7fec5c3d99c41ef7ceccb05aa2327188c19e3b30d9...


In [4]:
lib = pd.read_csv(ROOT / "data/processed/library.csv", dtype=str, keep_default_na=False, low_memory=False)
assert len(lib) == 19611
assert lib["isil"].nunique() == 19611
print("OK: master ICCU = 19.611 record, ISIL tutti unici.")

OK: master ICCU = 19.611 record, ISIL tutti unici.
